# Lattice Event Classification — Cross-Simulation Analysis

Loads the reference tables of every run of one composition (default `100Ni`,
NiAlH potential; set in the Parameters block of the next cell), classifies
events under 1.2 eV into lattice move types (1NN hop, 2NN hop, exchange,
concerted), and extracts parameters for an on-lattice KMC model.

The data root comes from `PYKMC_ANALYSIS_DATA_ROOT` (runs laid out as
`{composition}/{N}vac/T{temp}/{variant}/`).

Uses functions from `lattice_map_events.py` for event classification and
lattice projection.

In [ ]:
# Cell 0: Configuration & Data Loading
#
# Run from an environment where the package is installed (pip install -e .
# from the repository root) so `Analysis` is importable from this directory.
# Set the parameters in the "Parameters" block below.
import pickle
import configparser
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Interactive widgets for figure toggles (visibility + colour scheme).
# If running this notebook headless (e.g. nbconvert --execute), the widgets
# simply render as static output of their default state.
import ipywidgets as widgets
from IPython.display import display

# Available matplotlib qualitative colormaps for widget dropdowns.
CMAP_CHOICES = ['custom', 'tab10', 'tab20', 'Set1', 'Set2', 'Set3', 'Dark2', 'Paired', 'viridis']


def _palette(cmap_name, keys):
    """Return a {key: hex_color} dict for the given cmap name and ordered keys."""
    if cmap_name == 'custom':
        return None  # caller falls back to its hand-picked dict
    cmap = cm.get_cmap(cmap_name, max(len(keys), 2))
    return {k: cmap(i) for i, k in enumerate(keys)}

# Classification API from Analysis/lattice_map_events.py
from Analysis import _paths
from Analysis._paths import BARRIER_CUTOFF, CLASSIFICATION_DIR
from Analysis.sim_discovery import parse_run_path
from Analysis.reference_table import event_types
from Analysis.lattice_map_events import (
    # Classification
    classify_event_mechanism,
    classify_event_from_row,
    compute_saddle_features,
    compute_thresholds_gmm,
    classify_event_adaptive,
    classify_event_enhanced,
    TYPE_1NN, TYPE_1NN_COOP, TYPE_1NN_TRIPLE,
    TYPE_2NN, TYPE_EXCHANGE, TYPE_EXCHANGE_UP, TYPE_EXCHANGE_DOWN,
    TYPE_EXCHANGE_LAT,
    TYPE_CONCERTED, TYPE_COMPLEX, TYPE_RELAXATION, LATTICE_TYPES,
    TYPE_INTERLAYER_HOP_UP, TYPE_INTERLAYER_HOP_DOWN, TYPE_INPLANE_COMPLEX,
    TYPE_SUBSURFACE_MIGRATION, TYPE_SUBSURFACE_HOP,
    TYPE_SUBSURFACE_1NN_HOP, TYPE_SUBSURFACE_OTHER,
    DEALLOYING_TYPES,
    # Z-layer-aware reclassification helpers
    get_surface_layer, get_event_surface_layers, reclassify_by_zlayer,
    coarsen_event_type,
    MOVE_THRESHOLD, DIST_TOLERANCE,
    NN1_DIST, NN2_DIST, A_NI,
    # Config helpers
    read_sim_config, check_potential, REQUIRED_POTENTIAL,
    # Lattice helpers
    group_z_layers,
    generate_ideal_sites,
    find_vacancy_sites,
    build_surface_neighbour_list,
    map_event_to_lattice,
    # Step log
    parse_pykmc_out,
    compute_kinetic_coverage,
    build_rate_lookup,
)

# ── Parameters ──
# COMPOSITION      directory under the data root to classify
#                  (runs laid out as {composition}/{N}vac/T{temp}/{variant}/).
# DATA_ROOT_PATH   None uses $PYKMC_ANALYSIS_DATA_ROOT; or set '/path/to/data'.
# POTENTIAL_FILTER keep runs whose input.in pair_coeff contains this string;
#                  '' keeps every run.
# Outputs go to $PYKMC_ANALYSIS_OUTPUT_DIR, else Analysis/lattice_event_classification/.
COMPOSITION = '100Ni'
DATA_ROOT_PATH = None
POTENTIAL_FILTER = REQUIRED_POTENTIAL

DATA_ROOT = _paths.data_root(DATA_ROOT_PATH) / COMPOSITION
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'composition directory not found: {DATA_ROOT} (check COMPOSITION)')
OUT_DIR = CLASSIFICATION_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# BARRIER_CUTOFF imported from Analysis._paths (KMCConfig) — single source.
kB = 8.617333e-5      # eV/K — Boltzmann constant (used by ml_descriptors and arrhenius)

# All event types for iteration
ALL_TYPES = [TYPE_1NN, TYPE_1NN_COOP, TYPE_1NN_TRIPLE, TYPE_2NN,
             TYPE_EXCHANGE_UP, TYPE_EXCHANGE_DOWN, TYPE_EXCHANGE_LAT,
             TYPE_CONCERTED, TYPE_INTERLAYER_HOP_UP, TYPE_INTERLAYER_HOP_DOWN,
             TYPE_INPLANE_COMPLEX, TYPE_COMPLEX, TYPE_RELAXATION,
             TYPE_SUBSURFACE_MIGRATION, TYPE_SUBSURFACE_HOP,
             TYPE_SUBSURFACE_1NN_HOP, TYPE_SUBSURFACE_OTHER]

print(f'Data root: {DATA_ROOT}')
print(f'Output:    {OUT_DIR}')
print(f'Barrier cutoff: {BARRIER_CUTOFF} eV')
print(f'Potential filter: {POTENTIAL_FILTER or "(none)"}')
print(f'1NN distance: {NN1_DIST:.3f} A, 2NN distance: {NN2_DIST:.3f} A')

In [ ]:
# Cell 0b: Scan and load the runs of COMPOSITION that pass POTENTIAL_FILTER

def scan_simulations(root, potential_filter=POTENTIAL_FILTER):
    """Find run directories under root ({N}vac/T{temp}/{variant}/).

    Keeps runs with an input.in whose pair_coeff contains potential_filter
    ('' keeps every run). The layout is parsed by parse_run_path.
    """
    sims = []
    for variant_dir in sorted(root.glob('*/*/*')):
        if not variant_dir.is_dir():
            continue
        parsed = parse_run_path(variant_dir)
        if parsed is None:
            continue
        if not (variant_dir / 'input.in').exists():
            continue
        _, nvac, temp, variant = parsed

        sim_config = read_sim_config(variant_dir)
        if not check_potential(sim_config, potential_filter):
            continue

        sims.append({
            'nvac': nvac,
            'temp': int(temp),
            'variant': variant,
            'path': variant_dir,
            'potential': sim_config['potential'],
        })
    return sims


sims = scan_simulations(DATA_ROOT)
print(f'Found {len(sims)} simulations under {DATA_ROOT}')

# Load all reference tables
all_events = []
load_failures = []

for sim in sims:
    ref_path = sim['path'] / 'reference_table.pickle'
    if not ref_path.exists():
        load_failures.append((sim, 'no pickle'))
        continue
    try:
        df = pd.read_pickle(ref_path)
        df = df.copy()
        df['nvac'] = sim['nvac']
        df['temp'] = sim['temp']
        df['sim_path'] = str(sim['path'])
        df['sim_label'] = f"{sim['nvac']}vac_T{sim['temp']}"
        all_events.append(df)
    except Exception as e:
        load_failures.append((sim, str(e)))

if load_failures:
    print(f'\nLoad failures ({len(load_failures)}):')
    for sim, err in load_failures:
        print(f'  {sim["nvac"]}vac T{sim["temp"]} {sim["variant"]}: {err}')

# Combine into master DataFrame
master = pd.concat(all_events, ignore_index=True)
print(f'\nTotal events loaded: {len(master)}')
print(f'Events under {BARRIER_CUTOFF} eV: {(master["energy_barrier"] < BARRIER_CUTOFF).sum()}')
print(f'\nPer-simulation summary:')
summary = master.groupby(['nvac', 'temp']).agg(
    n_total=('energy_barrier', 'count'),
    n_sub_cutoff=('energy_barrier', lambda x: (x < BARRIER_CUTOFF).sum()),
    ea_min=('energy_barrier', 'min'),
    ea_max=('energy_barrier', 'max'),
).reset_index()
print(summary.to_string(index=False))

In [ ]:
# Cell 1: Coverage Heatmap

# Build coverage matrix
nvac_vals = sorted(master['nvac'].unique())
temp_vals = sorted(master['temp'].unique())

n_total_grid = np.full((len(nvac_vals), len(temp_vals)), np.nan)
n_sub_grid = np.full((len(nvac_vals), len(temp_vals)), np.nan)

for i, nv in enumerate(nvac_vals):
    for j, t in enumerate(temp_vals):
        mask = (master['nvac'] == nv) & (master['temp'] == t)
        if mask.any():
            n_total_grid[i, j] = mask.sum()
            n_sub_grid[i, j] = (master.loc[mask, 'energy_barrier'] < BARRIER_CUTOFF).sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, grid, title in [
    (axes[0], n_total_grid, 'Total events in catalogue'),
    (axes[1], n_sub_grid, f'Events under {BARRIER_CUTOFF} eV'),
]:
    im = ax.imshow(grid, aspect='auto', origin='lower',
                   cmap='YlOrRd', interpolation='nearest')
    ax.set_xticks(range(len(temp_vals)))
    ax.set_xticklabels([f'T{t}' for t in temp_vals], rotation=45)
    ax.set_yticks(range(len(nvac_vals)))
    ax.set_yticklabels([f'{n}vac' for n in nvac_vals])
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('Vacancy count')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='N events')

    # Annotate cells
    for i in range(len(nvac_vals)):
        for j in range(len(temp_vals)):
            val = grid[i, j]
            if np.isnan(val):
                ax.text(j, i, '—', ha='center', va='center', fontsize=9, color='grey')
            else:
                ax.text(j, i, f'{int(val)}', ha='center', va='center', fontsize=8,
                        color='white' if val > grid[~np.isnan(grid)].max() * 0.6 else 'black')

plt.suptitle('NiAlH_jea.eam — Ni(100) Event Catalogue Coverage', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'coverage_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Report missing cells
all_temps = [300, 400, 500, 600, 700, 800]
all_nvacs = list(range(1, 11))
print('\nMissing simulations (T300-T800 grid):')
for nv in all_nvacs:
    missing = [t for t in all_temps
               if not ((master['nvac'] == nv) & (master['temp'] == t)).any()]
    if missing:
        print(f'  {nv}vac: {[", ".join(f"T{t}" for t in missing)]}')

In [ ]:
# Cell 2: Classification Engine
#
# Uses classify_event_from_row from lattice_map_events.py — shared with
# the per-simulation script so classification is consistent.
# Now includes saddle-point features for downstream analysis.

# Apply classification to all sub-cutoff events
sub = master[master['energy_barrier'] < BARRIER_CUTOFF].copy()
print(f'Classifying {len(sub)} events under {BARRIER_CUTOFF} eV (with saddle features)...')

classifications = []
for idx, row in sub.iterrows():
    cls = classify_event_from_row(row, include_saddle_features=True)
    cls['master_idx'] = idx
    classifications.append(cls)

cls_df = pd.DataFrame(classifications).set_index('master_idx')

# Merge back
sub = sub.join(cls_df)

# Rename for consistency with rest of notebook
sub.rename(columns={'event_type': 'move_type'}, inplace=True)


def _base_move_type(mt: str) -> str:
    """Collapse forward/backward direction suffixes.

    Up and down variants of the same saddle are the forward/backward
    labels of a single event; for visualisation and rate aggregation the
    directionless base type is usually what you want. Direction is
    preserved in ``move_dz_signed`` so the merge is lossless.
    """
    if mt.endswith('_up') or mt.endswith('_down'):
        return mt.rsplit('_', 1)[0]
    return mt


sub['move_type_base'] = sub['move_type'].map(_base_move_type)

print(f'\nClassification results:')
type_counts = sub['move_type'].value_counts()
for mtype, count in type_counts.items():
    pct = 100 * count / len(sub)
    ea_mean = sub.loc[sub['move_type'] == mtype, 'energy_barrier'].mean()
    ea_std = sub.loc[sub['move_type'] == mtype, 'energy_barrier'].std()
    on_lat = ' [ON-LATTICE]' if mtype in LATTICE_TYPES else ''
    print(f'  {mtype:20s}: {count:5d} ({pct:5.1f}%)  '
          f'Ea = {ea_mean:.3f} \u00b1 {ea_std:.3f} eV{on_lat}')

# Flag complex events for review
n_complex = (sub['move_type'] == TYPE_COMPLEX).sum()
if n_complex > 0:
    print(f'\n{n_complex} events classified as "{TYPE_COMPLEX}" \u2014 review needed:')
    for _, row in sub[sub['move_type'] == TYPE_COMPLEX].head(10).iterrows():
        print(f'  {row["sim_label"]} event {row.name}: '
              f'Ea={row["energy_barrier"]:.3f} eV, '
              f'd={row["move_dist"]:.3f} A, dz={row["move_dz"]:.3f} A, '
              f'n_moved={row["n_moved"]}')

In [ ]:
# Cell 4b: ML Descriptors
#
# Adds a block of features designed for training barrier-prediction ML models:
#   Tier 1  mover-only      (disp_top_k, mover depth, species hist 1NN+2NN, Ea/kT)
#   Tier 2  neighborhood    (coordination, nearby vacancies, bond lengths, RMSD)
#   Tier 3  saddle-path     (path straightness, bond breaking, coord at saddle)
#   Tier 4  metadata        (reverse barrier, convergence flags, KMC fire count)
#
# All descriptors are computed at notebook-load time from data already in the
# per-event row (positions, species, move_atom_idx, energy_barrier) plus
# per-simulation cell vectors loaded once from trajkmc.xyz frame 0 and step
# logs parsed once from pykmc.out. No reference_table.pickle is regenerated.

from collections import Counter

from ase.io import read as ase_read

from Analysis.lattice_map_events import (
    compute_env_fingerprint,
    compute_ml_descriptors,
    parse_pykmc_out,
)

COMPUTE_ML_DESCRIPTORS = True
# Lattice adjacency (build_surface_neighbour_list) is expensive
# (~minutes per sim due to its pure-Python O(surface × sites × 9PBC) loop).
# Required for Tier 2 features `n_vac_nn1_initial/final`. With the flag off
# they fall back to a bulk coordination-shortfall proxy (wrong for surface
# atoms) — correct the exported CSV with `python -m Analysis.tools.fix_catalogue_nvac`
# (see Analysis/tools/README.md). The Stage 1h environment fingerprint runs
# from cluster geometry alone — no lattice adjacency needed — so it always fires.
COMPUTE_LATTICE_ADJACENCY = False

if COMPUTE_ML_DESCRIPTORS:
    # ── Per-simulation caches: load cells, lattices, step logs ──
    sim_cells = {}
    sim_lattices = {}   # sim_path -> (ideal_sites, neighbours_dict)
    sim_firing = {}     # sim_path -> Counter[ref_event] -> #selections

    for sp in sub['sim_path'].unique():
        sp_path = Path(sp)

        # Cell vectors from trajectory frame 0.
        # generate_ideal_sites and build_surface_neighbour_list expect cell as
        # a 3-element scalar array [Lx, Ly, Lz] (the diagonal), not the full 3x3
        # matrix (same convention as build_lattice_infrastructure in
        # Analysis/build_rate_table.py).
        try:
            atoms0 = ase_read(sp_path / 'trajkmc.xyz', index=0)
            sim_cells[sp] = list(atoms0.cell.array.diagonal())
        except Exception as e:
            print(f"[ml] cell load failed for {sp_path.name}: {e}")
            sim_cells[sp] = None

        # Ideal lattice + 1NN adjacency dict (expensive; gated by flag)
        if COMPUTE_LATTICE_ADJACENCY and sim_cells[sp] is not None:
            try:
                nvac_sp = int(sub.loc[sub['sim_path'] == sp, 'nvac'].iloc[0])
                cell_diag = sim_cells[sp]
                ideal = generate_ideal_sites(
                    atoms0.get_positions(), cell_diag, nvac_sp
                )
                neighbours, _, _, _, _ = build_surface_neighbour_list(
                    ideal, cell_diag, NN1_DIST
                )
                sim_lattices[sp] = (ideal, neighbours)
            except Exception as e:
                print(f"[ml] lattice load failed for {sp_path.name}: {e}")
                sim_lattices[sp] = (None, None)
        else:
            sim_lattices[sp] = (None, None)

        # KMC fire count from step log
        pykmc_path = sp_path / 'pykmc.out'
        if pykmc_path.exists():
            try:
                df_steps = parse_pykmc_out(pykmc_path)
                if len(df_steps) > 0:
                    sim_firing[sp] = Counter(df_steps['ref_event'].astype(int).tolist())
                else:
                    sim_firing[sp] = Counter()
            except Exception as e:
                print(f"[ml] firing-log parse failed for {sp_path.name}: {e}")
                sim_firing[sp] = Counter()
        else:
            sim_firing[sp] = Counter()

    print(f"[ml] cached {len(sim_cells)} sims; "
          f"{sum(1 for v in sim_cells.values() if v is not None)} with valid cell, "
          f"{sum(1 for v in sim_firing.values() if v)} with non-empty firing log"
          + (f", {sum(1 for v in sim_lattices.values() if v[0] is not None)} with lattice"
             if COMPUTE_LATTICE_ADJACENCY else ""))

    # ── Per-event loop: compute ML descriptors + Stage 1h fingerprint ──
    ml_rows = []
    n_fp_complete = 0
    for idx, row in sub.iterrows():
        sp = row['sim_path']
        cell = sim_cells.get(sp)
        ideal, adj = sim_lattices.get(sp, (None, None))
        fire = sim_firing.get(sp, Counter())

        features = compute_ml_descriptors(
            row, cell=cell, ideal_sites=ideal, nn_adjacency=adj
        )
        features['master_idx'] = idx
        features['kmc_fire_count'] = int(fire.get(int(row['idx_ref']), 0))

        # Stage 1h: lattice-environment fingerprint (cluster-based, no global lattice)
        fp = compute_env_fingerprint(row)
        features.update(fp)
        if fp['env_shell_complete']:
            n_fp_complete += 1

        ml_rows.append(features)

    ml_df = pd.DataFrame(ml_rows).set_index('master_idx')
    print(f"[fp] complete shells: {n_fp_complete}/{len(sub)} "
          f"({100*n_fp_complete/len(sub):.1f}%)")

    # ── Reverse-barrier self-join on (sim_path, idx_backward → idx_ref) ──
    # idx_backward is per-sim (not global master index), so key on both.
    rev_lookup = sub[['sim_path', 'idx_ref', 'energy_barrier']].rename(
        columns={'idx_ref': 'idx_backward', 'energy_barrier': 'reverse_barrier'}
    )
    pairs = sub[['sim_path', 'idx_backward']].merge(
        rev_lookup, on=['sim_path', 'idx_backward'], how='left'
    )
    ml_df['reverse_barrier'] = pairs['reverse_barrier'].values
    ml_df['Ea_gap'] = sub['energy_barrier'].values - ml_df['reverse_barrier'].values
    n_paired = int(ml_df['reverse_barrier'].notna().sum())
    print(f"[ml] reverse-barrier paired {n_paired}/{len(ml_df)} events "
          f"({100*n_paired/len(ml_df):.1f}%)")

    # ── Merge into sub ──
    sub = sub.join(ml_df, rsuffix='_ml')
    print(f"[ml] added {ml_df.shape[1]} ML columns to sub ({len(sub)} rows)")

In [ ]:
# Cell 2b: Saddle Feature Distributions
#
# Visualises saddle-point geometry features to validate the
# hop-vs-exchange discriminator (saddle_height_ratio).
#
# NOTE: this cell runs BEFORE the z-layer pass (cell 07), so
# `move_type` here still carries the pre-pass displacement-based
# labels (exchange_up/down, etc.). Post-pass subsurface labels only
# appear from cell 07 onward.

saddle_cols = ['saddle_disp_mag', 'saddle_dz', 'saddle_height_ratio',
               'saddle_n_displaced', 'saddle_asymmetry', 'saddle_max_pair_dz']

has_saddle = sub['saddle_height_ratio'].notna()
print(f'Events with saddle features: {has_saddle.sum()} / {len(sub)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Box plot: saddle_height_ratio by move_type
ax = axes[0]
types_for_box = [t for t in ALL_TYPES
                 if t in sub.loc[has_saddle, 'move_type'].unique()]
data = [sub.loc[has_saddle & (sub['move_type'] == t), 'saddle_height_ratio'].dropna().values
        for t in types_for_box]
data = [d for d, t in zip(data, types_for_box) if len(d) > 0]
types_for_box = [t for t, d_orig in zip(types_for_box,
    [sub.loc[has_saddle & (sub['move_type'] == t), 'saddle_height_ratio'].dropna().values
     for t in types_for_box]) if len(d_orig) > 0]
if data:
    ax.boxplot(data, tick_labels=[t.replace('_', '\n') for t in types_for_box],
               patch_artist=True, showfliers=False, whis=[5, 95])
    ax.axhline(1.15, color='red', ls='--', alpha=0.7, label='exchange threshold (1.15)')
    ax.set_ylabel('saddle_height_ratio')
    ax.set_title('Saddle height ratio by event type')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

# 2. Scatter: saddle_height_ratio vs saddle_max_pair_dz, colored by move_type
ax = axes[1]
for t in types_for_box:
    mask = has_saddle & (sub['move_type'] == t)
    ax.scatter(sub.loc[mask, 'saddle_max_pair_dz'],
               sub.loc[mask, 'saddle_height_ratio'],
               label=t, alpha=0.4, s=10)
ax.axhline(1.15, color='red', ls='--', alpha=0.7)
ax.set_xlabel('saddle_max_pair_dz (\u00c5)')
ax.set_ylabel('saddle_height_ratio')
ax.set_title('Saddle geometry: height ratio vs max pair dz')
ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'saddle_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Export saddle feature distributions
saddle_export = sub.loc[has_saddle, ['sim_label', 'idx_ref', 'move_type',
                                     'energy_barrier'] + saddle_cols].copy()
saddle_export.to_csv(OUT_DIR / 'saddle_feature_distributions.csv', index=False)
print(f'Saved saddle_feature_distributions.csv ({len(saddle_export)} events)')

In [ ]:
# Cell 2c: GMM Adaptive Threshold Comparison
#
# Fits a Gaussian Mixture Model to displacement distributions to derive
# adaptive thresholds, and compares them to the hardcoded values.
#
# NOTE: this cell runs BEFORE the z-layer pass (cell 07); `move_type`
# still carries the pre-pass labels. Reclassification tallies here
# refer to GMM-threshold vs hard-coded threshold changes, not the
# z-layer pass.

# Compute GMM thresholds from the classified events
displacements = sub['move_dist'].values
dz_values = sub['move_dz'].values

thresholds_gmm = compute_thresholds_gmm(displacements, dz_values, lattice_constant=A_NI)

print('GMM-derived thresholds vs hardcoded:')
print(f'  1NN centre:  {thresholds_gmm["nn1_center"]:.4f} A  (hardcoded: {NN1_DIST:.4f} A)')
print(f'  2NN centre:  {thresholds_gmm["nn2_center"]:.4f} A  (hardcoded: {NN2_DIST:.4f} A)')
print(f'  1NN range:   [{thresholds_gmm["nn1_range"][0]:.3f}, {thresholds_gmm["nn1_range"][1]:.3f}] A  '
      f'(hardcoded: [{NN1_DIST - DIST_TOLERANCE:.3f}, {NN1_DIST + DIST_TOLERANCE:.3f}] A)')
print(f'  2NN range:   [{thresholds_gmm["nn2_range"][0]:.3f}, {thresholds_gmm["nn2_range"][1]:.3f}] A  '
      f'(hardcoded: [{NN2_DIST - DIST_TOLERANCE:.3f}, {NN2_DIST + DIST_TOLERANCE:.3f}] A)')
print(f'  Layer change: {thresholds_gmm["layer_change_threshold"]:.3f} A  (hardcoded: 1.000 A)')
print(f'  GMM confidence (Bhattacharyya): {thresholds_gmm["confidence"]:.2f}')

# Plot GMM components overlaid on displacement histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Displacement GMM
ax = axes[0]
inplane = sub[sub['move_dz'].abs() < 1.0]
d_vals = inplane.loc[inplane['move_dist'] > MOVE_THRESHOLD, 'move_dist'].values
ax.hist(d_vals, bins=80, density=True, alpha=0.5, color='steelblue', label='data')
x = np.linspace(0, 5, 500)
gmm = thresholds_gmm['gmm_displacement']
for i, (w, m, v) in enumerate(zip(gmm.weights_, gmm.means_.ravel(),
                                    gmm.covariances_.ravel())):
    ax.plot(x, w * np.exp(-0.5 * (x - m)**2 / v) / np.sqrt(2 * np.pi * v),
            '--', lw=2, label=f'Component {i+1}: \u03bc={m:.3f}')
ax.axvline(NN1_DIST, color='green', ls=':', alpha=0.8, label=f'1NN={NN1_DIST:.3f}')
ax.axvline(NN2_DIST, color='red', ls=':', alpha=0.8, label=f'2NN={NN2_DIST:.3f}')
ax.set_xlabel('Displacement (\u00c5)')
ax.set_ylabel('Density')
ax.set_title('Displacement GMM (in-plane events)')
ax.legend(fontsize=7)

# dz GMM
ax = axes[1]
abs_dz = sub['move_dz'].abs().values
abs_dz_plot = abs_dz[abs_dz > 0.01]
ax.hist(abs_dz_plot, bins=80, density=True, alpha=0.5, color='steelblue', label='data')
gmm_dz = thresholds_gmm['gmm_dz']
for i, (w, m, v) in enumerate(zip(gmm_dz.weights_, gmm_dz.means_.ravel(),
                                    gmm_dz.covariances_.ravel())):
    ax.plot(x, w * np.exp(-0.5 * (x - m)**2 / v) / np.sqrt(2 * np.pi * v),
            '--', lw=2, label=f'Component {i+1}: \u03bc={m:.3f}')
ax.axvline(thresholds_gmm['layer_change_threshold'], color='red', ls='--',
           label=f'layer thr={thresholds_gmm["layer_change_threshold"]:.3f}')
ax.axvline(1.0, color='green', ls=':', label='hardcoded thr=1.0')
ax.set_xlabel('|dz| (\u00c5)')
ax.set_ylabel('Density')
ax.set_title('|dz| GMM (layer change detection)')
ax.legend(fontsize=7)
ax.set_xlim(0, 3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'gmm_threshold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Reclassify with adaptive thresholds and report differences
adaptive_types = []
for idx, row in sub.iterrows():
    cls = classify_event_adaptive(row, thresholds_gmm)
    adaptive_types.append(cls['event_type'])

sub['adaptive_type'] = adaptive_types
changed = sub['move_type'] != sub['adaptive_type']
print(f'\nAdaptive reclassification: {changed.sum()} / {len(sub)} events changed ({100*changed.mean():.2f}%)')
if changed.any():
    print('Changes by type:')
    for old_t in sub.loc[changed, 'move_type'].unique():
        new_types = sub.loc[changed & (sub['move_type'] == old_t), 'adaptive_type'].value_counts()
        for new_t, n in new_types.items():
            print(f'  {old_t} -> {new_t}: {n}')

# Save adaptive thresholds
import json as json_mod
thr_export = {k: v for k, v in thresholds_gmm.items()
              if k not in ('gmm_displacement', 'gmm_dz')}
# Convert tuples to lists for JSON
thr_export['nn1_range'] = list(thr_export['nn1_range'])
thr_export['nn2_range'] = list(thr_export['nn2_range'])
with open(OUT_DIR / 'adaptive_thresholds.json', 'w') as f:
    json_mod.dump(thr_export, f, indent=2)
print(f'Saved adaptive_thresholds.json')

In [ ]:
# Cell 2d: Enhanced Classification (Saddle-Assisted)
#
# Uses classify_event_enhanced to resolve ambiguous boundary events
# using saddle-point geometry.

print('Running enhanced classification (saddle-assisted)...')
enhanced_results = []
for idx, row in sub.iterrows():
    cls = classify_event_enhanced(row, thresholds=thresholds_gmm)
    cls['master_idx'] = idx
    enhanced_results.append(cls)

enh_df = pd.DataFrame(enhanced_results).set_index('master_idx')
sub['enhanced_type'] = enh_df['event_type']
sub['reclassified'] = enh_df['reclassified']
sub['original_event_type'] = enh_df['original_event_type']
sub['likely_reverse_relaxation'] = enh_df['likely_reverse_relaxation']

n_reclass = sub['reclassified'].sum()
print(f'\nEnhanced reclassification: {n_reclass} / {len(sub)} events ({100*n_reclass/len(sub):.2f}%)')

if n_reclass > 0:
    print('\nReclassification summary:')
    reclass = sub[sub['reclassified']]
    for _, row in reclass.groupby(['original_event_type', 'enhanced_type']).size().reset_index(name='count').iterrows():
        print(f'  {row["original_event_type"]} -> {row["enhanced_type"]}: {row["count"]}')

    # Barrier distributions of reclassified events
    print(f'\nReclassified event barriers:')
    print(f'  Mean Ea: {reclass["energy_barrier"].mean():.3f} eV')
    print(f'  Std Ea:  {reclass["energy_barrier"].std():.3f} eV')
    print(f'  Range:   [{reclass["energy_barrier"].min():.3f}, {reclass["energy_barrier"].max():.3f}] eV')

    # Displacement scatter with reclassified events highlighted
    fig, ax = plt.subplots(figsize=(8, 6))
    not_reclass = sub[~sub['reclassified']]
    ax.scatter(not_reclass['move_dist'], not_reclass['move_dz'],
               alpha=0.2, s=5, c='grey', label='unchanged')
    ax.scatter(reclass['move_dist'], reclass['move_dz'],
               alpha=0.8, s=20, c='red', marker='x', label=f'reclassified (n={n_reclass})')
    ax.axhline(1.0, color='blue', ls='--', alpha=0.5, label='layer_change=1.0 A')
    ax.axvline(NN1_DIST, color='green', ls=':', alpha=0.5, label=f'1NN={NN1_DIST:.2f}')
    ax.axvline(NN2_DIST, color='orange', ls=':', alpha=0.5, label=f'2NN={NN2_DIST:.2f}')
    ax.set_xlabel('|displacement| (A)')
    ax.set_ylabel('|dz| (A)')
    ax.set_title('Enhanced classification: reclassified events near boundaries')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'enhanced_reclassification.png', dpi=150, bbox_inches='tight')
    plt.show()

# Report reverse-relaxation events
n_rev_relax = sub['likely_reverse_relaxation'].sum()
if n_rev_relax > 0:
    print(f'\nNear-zero barrier interlayer_hop_down -> relaxation: {n_rev_relax} events')


# ── Z-layer pass ───────────────────────────────────────────────────────────
# Relabel events whose primary moving atom sits below the top surface layer.
# This catches subsurface vacancy migrations that displacement-only rules
# misclassify as exchange_up/down (mover sits in layer 1+ but has big |dz|).

print('\nRunning z-layer reclassification...')
_layer_pairs = [
    get_event_surface_layers(row['initial_positions'], row['final_positions'],
                             int(row['move_atom_idx']))
    for _, row in sub.iterrows()
]
sub['surface_layer_initial'] = [p[0] for p in _layer_pairs]
sub['surface_layer_final'] = [p[1] for p in _layer_pairs]
# Keep `surface_layer` as an alias of the initial layer for convenience /
# backward compatibility.
sub['surface_layer'] = sub['surface_layer_initial']
sub['move_type_pre_zlayer'] = sub['move_type']
sub['move_type'] = [
    reclassify_by_zlayer(mt, li, lf)
    for mt, li, lf in zip(sub['move_type'],
                          sub['surface_layer_initial'],
                          sub['surface_layer_final'])
]
sub['zlayer_reclassified'] = sub['move_type'] != sub['move_type_pre_zlayer']
# Refresh the directionless alias introduced in 04_load_events so downstream
# cells and the interactive widgets immediately pick up subsurface labels.
sub['move_type_base'] = sub['move_type'].map(_base_move_type)

n_zre = int(sub['zlayer_reclassified'].sum())
print(f'Z-layer reclassification: {n_zre}/{len(sub)} events '
      f'({100*n_zre/len(sub):.3f}%)')
if n_zre:
    print('  by (from -> to):')
    grp = (sub[sub['zlayer_reclassified']]
             .groupby(['move_type_pre_zlayer', 'move_type'])
             .size()
             .sort_values(ascending=False))
    for (old, new), n in grp.items():
        print(f'    {old:22s} -> {new:24s} n={n}')

print('\nEvents by (layer_initial, layer_final)  [0 = top surface]:')
print(sub.groupby(['surface_layer_initial', 'surface_layer_final'])
         .size().sort_index().to_string())

# ── Floating-atom reclassification (post z-layer) ─────────────────────────
# interlayer_hop_down with |d| > 3 Å are atoms that detached from the surface
# entirely — pARTn found the saddle but the event is not a physical hop.
# These have zero-barrier return paths and are reclassified as relaxation.
_float_mask = (
    (sub['move_type'] == 'interlayer_hop_down')
    & (sub['move_dist'] > 3.0)
)
_n_float = int(_float_mask.sum())
if _n_float > 0:
    sub.loc[_float_mask, 'move_type'] = 'relaxation'
    sub.loc[_float_mask, 'move_type_base'] = 'relaxation'
    sub.loc[_float_mask, 'zlayer_reclassified'] = True
    print(f'\nFloating-atom reclassification: {_n_float} interlayer_hop_down '
          f'with |d|>3 A -> relaxation')
else:
    print(f'\nNo floating-atom events found (interlayer_hop_down |d|>3 A)')

In [ ]:
# Cell 2e: Species Breakdown
#
# For multi-element simulations: barrier distributions by mover species.
# For pure Ni, this confirms all movers are type 1 (Ni).

# Check if species info is available
has_types = False
for idx, row in sub.head(5).iterrows():
    if event_types(row) is not None:
        has_types = True
        break

if has_types:
    print('Extracting species information...')
    species_info = []
    for idx, row in sub.iterrows():
        sp = classify_event_from_row(row, include_species=True)
        species_info.append({
            'master_idx': idx,
            'mover_species': sp.get('mover_species'),
            'partner_species': sp.get('partner_species'),
        })
    sp_df = pd.DataFrame(species_info).set_index('master_idx')
    sub['mover_species'] = sp_df['mover_species']
    sub['partner_species'] = sp_df['partner_species']

    unique_species = sub['mover_species'].dropna().unique()
    print(f'Unique mover species: {unique_species}')
    print(f'\nSpecies breakdown by move type:')
    species_summary = sub.groupby(['move_type', 'mover_species']).agg(
        count=('energy_barrier', 'size'),
        Ea_mean=('energy_barrier', 'mean'),
        Ea_std=('energy_barrier', 'std'),
    ).reset_index()
    print(species_summary.to_string(index=False))

    # If multiple species, plot barrier distributions by species
    if len(unique_species) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        for sp in sorted(unique_species):
            sp_data = sub[sub['mover_species'] == sp]
            ax.hist(sp_data['energy_barrier'], bins=50, alpha=0.5, label=f'Species {sp}')
        ax.set_xlabel('Energy barrier (eV)')
        ax.set_ylabel('Count')
        ax.set_title('Barrier distributions by mover species')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'species_barrier_distributions.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('No species information available in reference tables.')
    print('Species-dependent analysis skipped.')

In [ ]:
# Cell 3: Displacement Scatter — validates classification boundaries
# Interactive: toggle each move-type on/off, change colour scheme,
# adjust point size. The static PNG (saved to disk) uses the defaults.

custom_colors = {
    TYPE_1NN: '#4DAF4A',
    TYPE_1NN_COOP: '#8DD35F',
    TYPE_1NN_TRIPLE: '#006400',
    TYPE_2NN: '#377EB8',
    TYPE_EXCHANGE_UP: '#E41A1C',
    TYPE_EXCHANGE_DOWN: '#FF6666',
    TYPE_EXCHANGE_LAT: '#FF7F00',
    TYPE_CONCERTED: '#984EA3',
    TYPE_INTERLAYER_HOP_UP: '#F781BF',
    TYPE_INTERLAYER_HOP_DOWN: '#A65628',
    TYPE_INPLANE_COMPLEX: '#BBBBBB',
    TYPE_RELAXATION: '#DDDDDD',
    TYPE_COMPLEX: '#999999',
    # Subsurface (z-layer reclassified) — distinct purple/blue palette so
    # they pop visually against the surface-event cloud.
    TYPE_SUBSURFACE_MIGRATION: '#6A3D9A',
    TYPE_SUBSURFACE_HOP: '#CAB2D6',
    TYPE_SUBSURFACE_1NN_HOP: '#1F78B4',
    TYPE_SUBSURFACE_OTHER: '#666666',
    # Merged (up/down collapsed) — used when the "Merge up/down" widget
    # toggle is on. Pick the 'up' colour as canonical for each family.
    'exchange': '#E41A1C',
    'interlayer_hop': '#F781BF',
}
# keep a module-level default palette so cell 10 etc. can still reference `colors`
colors = custom_colors

# Directionless base types (used when 'Merge up/down' toggle is active).
ALL_TYPES_BASE = []
for _t in ALL_TYPES:
    _b = _base_move_type(_t)
    if _b not in ALL_TYPES_BASE:
        ALL_TYPES_BASE.append(_b)


def _plot_displacement(sub, active_types, cmap_name, point_size,
                       show_thresholds, merge_updown=False, save=False):
    type_col = 'move_type_base' if merge_updown else 'move_type'
    type_order = ALL_TYPES_BASE if merge_updown else ALL_TYPES
    palette = _palette(cmap_name, type_order) or custom_colors
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: |d| vs |dz|
    ax = axes[0]
    for mtype in type_order:
        if mtype not in active_types:
            continue
        mask = sub[type_col] == mtype
        if mask.any():
            ax.scatter(sub.loc[mask, 'move_dist'], sub.loc[mask, 'move_dz'],
                       color=palette.get(mtype, '#999'),
                       label=f'{mtype} ({mask.sum()})',
                       alpha=0.4, s=point_size, edgecolors='none')
    if show_thresholds:
        ax.axhline(1.0, color='grey', ls=':', lw=1, label='layer change threshold')
        ax.axvline(NN1_DIST, color='green', ls='--', lw=1, alpha=0.5,
                   label=f'1NN = {NN1_DIST:.2f} A')
        ax.axvline(NN2_DIST, color='blue', ls='--', lw=1, alpha=0.5,
                   label=f'2NN = {NN2_DIST:.2f} A')
    ax.set_xlabel('Primary atom displacement |d| (A)')
    ax.set_ylabel('Out-of-plane |dz| (A)')
    ax.set_title('Classification by displacement')
    ax.legend(fontsize=8, loc='upper right')

    # Plot 2: histogram
    ax = axes[1]
    for mtype in type_order:
        if mtype not in active_types:
            continue
        mask = sub[type_col] == mtype
        if mask.any():
            ax.hist(sub.loc[mask, 'move_dist'], bins=60, alpha=0.6,
                    color=palette.get(mtype, '#999'), label=mtype)
    if show_thresholds:
        ax.axvline(NN1_DIST, color='green', ls='--', lw=2,
                   label=f'1NN = {NN1_DIST:.2f} A')
        ax.axvline(NN2_DIST, color='blue', ls='--', lw=2,
                   label=f'2NN = {NN2_DIST:.2f} A')
    ax.set_xlabel('Primary atom displacement |d| (A)')
    ax.set_ylabel('Count')
    ax.set_title('Displacement distribution by move type')
    ax.legend(fontsize=8)

    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / 'displacement_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()


# Save the canonical static version first (so PNG output matches prior runs).
_plot_displacement(sub, set(ALL_TYPES), 'custom', 15, True, save=True)

# Interactive controller
# Two parallel checkbox grids — one for split types, one for base types —
# so the widget layout remains stable when 'Merge up/down' is toggled.
_split_toggles = {mt: widgets.Checkbox(value=True, description=mt,
                                       indent=False,
                                       layout=widgets.Layout(width='180px'))
                  for mt in ALL_TYPES}
_base_toggles = {mt: widgets.Checkbox(value=True, description=mt,
                                      indent=False,
                                      layout=widgets.Layout(width='180px'))
                 for mt in ALL_TYPES_BASE}
_cmap_dd = widgets.Dropdown(options=CMAP_CHOICES, value='custom',
                            description='Colours:')
_psize = widgets.IntSlider(value=15, min=2, max=40, step=1, description='Point size:')
_thresh = widgets.Checkbox(value=True, description='Show thresholds')
_merge = widgets.Checkbox(value=True, description='Merge up/down')


def _go(**kw):
    merge = kw['merge']
    if merge:
        active = {mt for mt in ALL_TYPES_BASE if kw['b_' + mt]}
    else:
        active = {mt for mt in ALL_TYPES if kw['s_' + mt]}
    _plot_displacement(sub, active, kw['cmap'], kw['psize'], kw['thresh'],
                       merge_updown=merge)


_controls = {'cmap': _cmap_dd, 'psize': _psize, 'thresh': _thresh,
             'merge': _merge,
             **{'s_' + mt: w for mt, w in _split_toggles.items()},
             **{'b_' + mt: w for mt, w in _base_toggles.items()}}
_ui = widgets.VBox([
    widgets.HTML('<b>Interactive view</b> — toggle types, colour scheme, point size:'),
    widgets.HBox([_cmap_dd, _psize, _thresh, _merge]),
    widgets.HTML('<i>base types (when Merge up/down is on):</i>'),
    widgets.GridBox(list(_base_toggles.values()),
                    layout=widgets.Layout(grid_template_columns='repeat(4, 180px)')),
    widgets.HTML('<i>full split types (when Merge up/down is off):</i>'),
    widgets.GridBox(list(_split_toggles.values()),
                    layout=widgets.Layout(grid_template_columns='repeat(4, 180px)')),
])
_out = widgets.interactive_output(_go, _controls)
display(_ui, _out)

In [ ]:
# Cell 4: Barrier Distributions by Type
# Interactive: toggle move types, switch colour scheme, change bin count, log-y.

_BARRIER_MTS_SPLIT = ['1NN_hop', '1NN_hop_cooperative', '2NN_hop',
                      'exchange_up', 'exchange_down', 'exchange_lateral',
                      'interlayer_hop_up', 'interlayer_hop_down',
                      'inplane_complex', 'complex',
                      'subsurface_migration', 'subsurface_hop',
                      'subsurface_1NN_hop', 'subsurface_other']
_BARRIER_MTS_BASE = ['1NN_hop', '1NN_hop_cooperative', '2NN_hop',
                     'exchange', 'exchange_lateral',
                     'interlayer_hop', 'inplane_complex', 'complex',
                     'subsurface_migration', 'subsurface_hop',
                     'subsurface_1NN_hop', 'subsurface_other']
# Back-compat alias for any code that still references _BARRIER_MTS
_BARRIER_MTS = _BARRIER_MTS_BASE


def _plot_barriers(sub, active_types, cmap_name, bins, log_y,
                   show_boxplot, merge_updown=True, save=False):
    type_col = 'move_type_base' if merge_updown else 'move_type'
    type_order = _BARRIER_MTS_BASE if merge_updown else _BARRIER_MTS_SPLIT
    present = [mt for mt in type_order if (sub[type_col] == mt).any()]
    palette = _palette(cmap_name, type_order) or colors
    mts = [mt for mt in present if mt in active_types]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Box plot by type
    ax = axes[0]
    if show_boxplot and mts:
        data_by_type = [sub.loc[sub[type_col] == mt, 'energy_barrier'].values
                        for mt in mts]
        bp = ax.boxplot(data_by_type, tick_labels=mts, patch_artist=True,
                        showfliers=True, whis=[5, 95])
        for patch, mt in zip(bp['boxes'], mts):
            patch.set_facecolor(palette.get(mt, '#999'))
            patch.set_alpha(0.7)
    ax.set_ylabel('Barrier Ea (eV)')
    ax.set_title('Barrier distribution by move type')
    ax.tick_params(axis='x', rotation=30)

    # Histogram overlay
    ax = axes[1]
    for mt in mts:
        vals = sub.loc[sub[type_col] == mt, 'energy_barrier']
        ax.hist(vals, bins=bins, alpha=0.5, color=palette.get(mt, '#999'),
                label=f'{mt} (n={len(vals)})', density=True)
    ax.set_xlabel('Barrier Ea (eV)')
    ax.set_ylabel('Density')
    if log_y:
        ax.set_yscale('log')
    ax.set_title('Barrier density by type')
    ax.legend(fontsize=8)

    # Box plot by nvac for 1NN hops (independent of merge toggle — 1NN_hop
    # has no up/down split)
    ax = axes[2]
    nn1_data = sub[sub['move_type'] == '1NN_hop']
    if len(nn1_data) > 0 and '1NN_hop' in active_types:
        nvac_groups = sorted(nn1_data['nvac'].unique())
        data_by_nvac = [nn1_data.loc[nn1_data['nvac'] == nv, 'energy_barrier'].values
                        for nv in nvac_groups]
        bp2 = ax.boxplot(data_by_nvac,
                         tick_labels=[f'{nv}vac' for nv in nvac_groups],
                         patch_artist=True, showfliers=True, whis=[5, 95])
        for patch in bp2['boxes']:
            patch.set_facecolor(palette.get('1NN_hop', '#4DAF4A'))
            patch.set_alpha(0.7)
        ax.set_ylabel('Barrier Ea (eV)')
        ax.set_title('1NN hop barriers by vacancy count')

    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / 'barrier_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()


# Canonical static save (merged up/down view — matches what we now
# consider the physically meaningful grouping)
_plot_barriers(sub, set(_BARRIER_MTS_BASE), 'custom', 40, False, True,
               merge_updown=True, save=True)

# Summary table
print('\nBarrier summary by move type (merged up/down):')
print(sub.groupby('move_type_base')['energy_barrier'].describe().round(4).to_string())

# Interactive controller
_bar_split_toggles = {mt: widgets.Checkbox(value=True, description=mt, indent=False,
                                           layout=widgets.Layout(width='170px'))
                      for mt in _BARRIER_MTS_SPLIT}
_bar_base_toggles = {mt: widgets.Checkbox(value=True, description=mt, indent=False,
                                          layout=widgets.Layout(width='170px'))
                     for mt in _BARRIER_MTS_BASE}
_bar_cmap = widgets.Dropdown(options=CMAP_CHOICES, value='custom', description='Colours:')
_bar_bins = widgets.IntSlider(value=40, min=10, max=120, step=5, description='Bins:')
_bar_logy = widgets.Checkbox(value=False, description='log y (hist)')
_bar_box = widgets.Checkbox(value=True, description='Show box plot')
_bar_merge = widgets.Checkbox(value=True, description='Merge up/down')


def _go_bar(**kw):
    merge = kw['merge']
    if merge:
        active = {mt for mt in _BARRIER_MTS_BASE if kw['b_' + mt]}
    else:
        active = {mt for mt in _BARRIER_MTS_SPLIT if kw['s_' + mt]}
    _plot_barriers(sub, active, kw['cmap'], kw['bins'], kw['logy'], kw['box'],
                   merge_updown=merge)


_bar_controls = {'cmap': _bar_cmap, 'bins': _bar_bins, 'logy': _bar_logy,
                 'box': _bar_box, 'merge': _bar_merge,
                 **{'s_' + mt: w for mt, w in _bar_split_toggles.items()},
                 **{'b_' + mt: w for mt, w in _bar_base_toggles.items()}}
_bar_ui = widgets.VBox([
    widgets.HTML('<b>Interactive view</b> — toggle types, colour scheme, bins:'),
    widgets.HBox([_bar_cmap, _bar_bins, _bar_logy, _bar_box, _bar_merge]),
    widgets.HTML('<i>base types (Merge on):</i>'),
    widgets.HBox(list(_bar_base_toggles.values())),
    widgets.HTML('<i>full split types (Merge off):</i>'),
    widgets.HBox(list(_bar_split_toggles.values())),
])
display(_bar_ui, widgets.interactive_output(_go_bar, _bar_controls))

In [ ]:
# Cell 5: Temperature Dependence
# Interactive: toggle which move types appear, colour scheme, error-bar
# display, and Arrhenius fit-line display.
# (kB is defined in 01_imports.py)

_ARR_MTS = ['1NN_hop', '2NN_hop', 'exchange', 'exchange_lateral']


def _plot_arrhenius(sub, active_types, cmap_name, show_err, show_fit, save=False):
    palette = _palette(cmap_name, _ARR_MTS) or colors
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean barrier vs temperature per type
    ax = axes[0]
    for mt in _ARR_MTS:
        if mt not in active_types:
            continue
        mt_data = sub[sub['move_type_base'] == mt]
        if len(mt_data) == 0:
            continue
        grouped = mt_data.groupby('temp')['energy_barrier'].agg(['mean', 'std', 'count'])
        grouped = grouped[grouped['count'] >= 3]
        if len(grouped) < 2:
            continue
        yerr = grouped['std'] if show_err else None
        ax.errorbar(grouped.index, grouped['mean'], yerr=yerr,
                    marker='o', label=mt, color=palette.get(mt, '#999'),
                    capsize=3, capthick=1)
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('Mean barrier Ea (eV)')
    ax.set_title('Mean barrier vs temperature')
    ax.legend()

    # Arrhenius plot
    ax = axes[1]
    if '1NN_hop' in active_types:
        nn1_temps = sub[sub['move_type_base'] == '1NN_hop'].groupby('temp')
        inv_t, ln_k = [], []
        for temp, grp in nn1_temps:
            if len(grp) < 3:
                continue
            k0 = 10.0
            ea_mean = grp['energy_barrier'].mean()
            k_mean = k0 * np.exp(-ea_mean / (kB * temp))
            inv_t.append(1000.0 / temp)
            ln_k.append(np.log(k_mean))
        if len(inv_t) >= 2:
            ax.plot(inv_t, ln_k, 'o-',
                    color=palette.get('1NN_hop', '#4DAF4A'), lw=2, markersize=8)
            if show_fit:
                coeffs = np.polyfit(inv_t, ln_k, 1)
                ea_eff = -coeffs[0] * kB * 1000
                ax.plot(inv_t, np.polyval(coeffs, inv_t), '--', color='grey',
                        label=f'Fit: Ea_eff = {ea_eff:.3f} eV')
                ax.legend()
    ax.set_xlabel('1000/T (1/K)')
    ax.set_ylabel('ln(k) [k in ps⁻¹]')
    ax.set_title('Arrhenius plot — 1NN hop rate')

    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / 'temperature_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()


# Canonical static save
_plot_arrhenius(sub, set(_ARR_MTS), 'custom', True, True, save=True)

# Interactive controller
_arr_toggles = {mt: widgets.Checkbox(value=True, description=mt, indent=False,
                                     layout=widgets.Layout(width='180px'))
                for mt in _ARR_MTS}
_arr_cmap = widgets.Dropdown(options=CMAP_CHOICES, value='custom', description='Colours:')
_arr_err = widgets.Checkbox(value=True, description='Error bars')
_arr_fit = widgets.Checkbox(value=True, description='Show fit')


def _go_arr(**kw):
    active = {mt for mt in _ARR_MTS if kw[mt]}
    _plot_arrhenius(sub, active, kw['cmap'], kw['err'], kw['fit'])


_arr_controls = {'cmap': _arr_cmap, 'err': _arr_err, 'fit': _arr_fit, **_arr_toggles}
_arr_ui = widgets.VBox([
    widgets.HTML('<b>Interactive view</b> — toggle types, errors, fit:'),
    widgets.HBox([_arr_cmap, _arr_err, _arr_fit]),
    widgets.HBox(list(_arr_toggles.values())),
])
display(_arr_ui, widgets.interactive_output(_go_arr, _arr_controls))

In [ ]:
# Cell 6: Vacancy Count Dependence
# Interactive: toggle which move types appear in both panels, colour scheme,
# stacked/grouped bar style.

_VAC_MTS = ['1NN_hop', '2NN_hop', 'exchange', 'exchange_lateral',
            'concerted', 'complex']


def _plot_vacancy(sub, active_types, cmap_name, bar_mode,
                  show_fraction, save=False):
    palette = _palette(cmap_name, _VAC_MTS) or colors
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    nvac_vals_present = sorted(sub['nvac'].unique())

    mts = [mt for mt in _VAC_MTS if mt in active_types]

    # Left: stacked or grouped bars of event counts by nvac
    ax = axes[0]
    x = np.arange(len(nvac_vals_present))
    if bar_mode == 'stacked':
        bottoms = np.zeros(len(nvac_vals_present))
        for mt in mts:
            counts = np.array([((sub['nvac'] == nv) & (sub['move_type_base'] == mt)).sum()
                               for nv in nvac_vals_present])
            if counts.sum() > 0:
                ax.bar(x, counts, bottom=bottoms,
                       color=palette.get(mt, '#999'), label=mt, alpha=0.85)
                bottoms += counts
    else:  # grouped
        n = max(len(mts), 1)
        w = 0.8 / n
        for i, mt in enumerate(mts):
            counts = np.array([((sub['nvac'] == nv) & (sub['move_type_base'] == mt)).sum()
                               for nv in nvac_vals_present])
            if counts.sum() > 0:
                ax.bar(x + (i - (n - 1) / 2) * w, counts, w,
                       color=palette.get(mt, '#999'), label=mt, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{nv}vac' for nv in nvac_vals_present])
    ax.set_ylabel('Event count')
    ax.set_title(f'Event types by vacancy count (Ea < {BARRIER_CUTOFF} eV)')
    ax.legend(fontsize=8, loc='upper left')

    # Right: fraction of each type vs nvac (if requested, else secondary count view)
    ax = axes[1]
    if show_fraction:
        for mt in mts:
            fracs = []
            for nv in nvac_vals_present:
                total = (sub['nvac'] == nv).sum()
                n_mt = ((sub['nvac'] == nv) & (sub['move_type_base'] == mt)).sum()
                fracs.append(100 * n_mt / total if total > 0 else 0)
            if max(fracs) > 0:
                ax.plot(nvac_vals_present, fracs, 'o-', color=palette.get(mt, '#999'),
                        label=mt, lw=2, markersize=6)
        ax.set_xlabel('Vacancy count')
        ax.set_ylabel('Fraction (%)')
        ax.set_title('Move type fraction vs vacancy count')
        ax.legend(fontsize=8)
        ax.set_ylim(0, 105)
    else:
        for mt in mts:
            counts = [((sub['nvac'] == nv) & (sub['move_type_base'] == mt)).sum()
                      for nv in nvac_vals_present]
            if sum(counts) > 0:
                ax.plot(nvac_vals_present, counts, 'o-',
                        color=palette.get(mt, '#999'), label=mt, lw=2, markersize=6)
        ax.set_xlabel('Vacancy count')
        ax.set_ylabel('Event count')
        ax.set_title('Event counts vs vacancy count')
        ax.legend(fontsize=8)

    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / 'vacancy_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()


# Canonical static save (stacked + fraction — matches the prior behaviour)
_plot_vacancy(sub, set(_VAC_MTS), 'custom', 'stacked', True, save=True)

# Interactive controller
_vac_toggles = {mt: widgets.Checkbox(value=True, description=mt, indent=False,
                                     layout=widgets.Layout(width='170px'))
                for mt in _VAC_MTS}
_vac_cmap = widgets.Dropdown(options=CMAP_CHOICES, value='custom', description='Colours:')
_vac_mode = widgets.ToggleButtons(options=['stacked', 'grouped'], value='stacked',
                                  description='Bars:')
_vac_frac = widgets.Checkbox(value=True, description='Right panel: fraction (%)')


def _go_vac(**kw):
    active = {mt for mt in _VAC_MTS if kw[mt]}
    _plot_vacancy(sub, active, kw['cmap'], kw['mode'], kw['frac'])


_vac_controls = {'cmap': _vac_cmap, 'mode': _vac_mode, 'frac': _vac_frac,
                 **_vac_toggles}
_vac_ui = widgets.VBox([
    widgets.HTML('<b>Interactive view</b> — toggle types, bar mode, fraction:'),
    widgets.HBox([_vac_cmap, _vac_mode, _vac_frac]),
    widgets.HBox(list(_vac_toggles.values())),
])
display(_vac_ui, widgets.interactive_output(_go_vac, _vac_controls))

In [ ]:
# Cell 7: Forward/Backward Pair Analysis
# Interactive: toggle scatter / type-mismatch panels, layers (diagonal
# reference, coloured by type), and colour scheme.

pair_rows = []
for sim_label in sub['sim_label'].unique():
    sim_events = sub[sub['sim_label'] == sim_label]
    for idx, row in sim_events.iterrows():
        bw_idx = row.get('idx_backward', -1)
        if bw_idx < 0:
            continue
        bw_match = sim_events[sim_events['idx_ref'] == bw_idx]
        if len(bw_match) == 0:
            continue
        bw_row = bw_match.iloc[0]
        pair_rows.append({
            'sim_label': sim_label,
            'nvac': row['nvac'],
            'temp': row['temp'],
            'fwd_idx': row['idx_ref'],
            'bwd_idx': bw_idx,
            'fwd_Ea': row['energy_barrier'],
            'bwd_Ea': bw_row['energy_barrier'],
            'fwd_type': row.get('move_type', 'unknown'),
            'bwd_type': bw_row.get('move_type', 'unknown'),
            'fwd_type_pre_zlayer': row.get('move_type_pre_zlayer', 'unknown'),
            'bwd_type_pre_zlayer': bw_row.get('move_type_pre_zlayer', 'unknown'),
            'fwd_reclass': bool(row.get('zlayer_reclassified', False)),
            'bwd_reclass': bool(bw_row.get('zlayer_reclassified', False)),
            'dE': row['energy_barrier'] - bw_row['energy_barrier'],
        })

pairs_df = pd.DataFrame(pair_rows)
print(f'Forward/backward pairs found: {len(pairs_df)}')


def _plot_pairs(pairs_df, cmap_name, colour_by_type, show_diagonal,
                show_scatter, show_mismatch, show_zlayer=True, save=False):
    if len(pairs_df) == 0:
        return
    mts = sorted(pairs_df['fwd_type'].unique())
    palette = _palette(cmap_name, mts) or colors

    n_panels = int(show_scatter) + int(show_mismatch) + int(show_zlayer)
    if n_panels == 0:
        return
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5),
                             squeeze=False)
    ax_iter = iter(axes[0])

    if show_scatter:
        ax = next(ax_iter)
        if colour_by_type:
            for mt in mts:
                mask = pairs_df['fwd_type'] == mt
                ax.scatter(pairs_df.loc[mask, 'fwd_Ea'],
                           pairs_df.loc[mask, 'bwd_Ea'],
                           color=palette.get(mt, '#999'), alpha=0.4, s=14,
                           label=f'{mt} ({mask.sum()})', edgecolors='none')
            ax.legend(fontsize=7, loc='upper left')
        else:
            ax.scatter(pairs_df['fwd_Ea'], pairs_df['bwd_Ea'],
                       alpha=0.3, s=10, c='steelblue')
        if show_diagonal:
            lim = max(pairs_df['fwd_Ea'].max(), pairs_df['bwd_Ea'].max()) + 0.1
            ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='symmetric')
        ax.set_xlabel('Forward Ea (eV)')
        ax.set_ylabel('Backward Ea (eV)')
        ax.set_title('Forward vs backward barriers')

    if show_mismatch:
        ax = next(ax_iter)
        same_type = (pairs_df['fwd_type'] == pairs_df['bwd_type']).sum()
        diff_type = len(pairs_df) - same_type
        ax.bar(['Same type', 'Different type'], [same_type, diff_type],
               color=['#4DAF4A', '#E41A1C'])
        ax.set_ylabel('Count')
        ax.set_title('Forward/backward type consistency (post z-layer)')
        ax.text(0, same_type + 5, f'{same_type}', ha='center', fontsize=12)
        ax.text(1, diff_type + 5, f'{diff_type}', ha='center', fontsize=12)

    if show_zlayer and 'fwd_reclass' in pairs_df.columns:
        ax = next(ax_iter)
        # Four mutually exclusive buckets describing whether the forward
        # and backward halves of a pair were touched by the z-layer pass.
        f, b = pairs_df['fwd_reclass'], pairs_df['bwd_reclass']
        n_neither = (~f & ~b).sum()
        n_both = (f & b).sum()
        n_fwd_only = (f & ~b).sum()
        n_bwd_only = (~f & b).sum()
        cats = ['Neither\nreclassified', 'Both\nreclassified',
                'Fwd only', 'Bwd only']
        vals = [int(n_neither), int(n_both), int(n_fwd_only), int(n_bwd_only)]
        bar_colors = ['#4DAF4A', '#6A3D9A', '#FFA500', '#FF7F00']
        ax.bar(cats, vals, color=bar_colors)
        ax.set_ylabel('Pair count')
        ax.set_title('Z-layer pass effect on pairs')
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(i, v + max(vals) * 0.01, f'{v}', ha='center', fontsize=10)

    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / 'pair_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()


if len(pairs_df) > 0:
    # Canonical static save (includes z-layer audit panel)
    _plot_pairs(pairs_df, 'custom', False, True, True, True,
                show_zlayer=True, save=True)

    # Report asymmetric pairs (unchanged)
    asym = pairs_df[pairs_df['fwd_type'] != pairs_df['bwd_type']]
    if len(asym) > 0:
        print(f'\nAsymmetric type pairs ({len(asym)} total):')
        type_combos = asym.groupby(['fwd_type', 'bwd_type']).size().sort_values(ascending=False)
        for (ft, bt), n in type_combos.items():
            print(f'  {ft} ↔ {bt}: {n}')

    # Report z-layer-split pairs (fwd and bwd on different sides of the
    # surface layer boundary — potential classifier edge cases).
    if 'fwd_reclass' in pairs_df.columns:
        split = pairs_df[pairs_df['fwd_reclass'] != pairs_df['bwd_reclass']]
        if len(split) > 0:
            print(f'\nZ-layer-split pairs ({len(split)} pairs where only '
                  f'one side was reclassified):')
            flip_combos = (split.groupby(['fwd_type_pre_zlayer', 'fwd_type',
                                          'bwd_type_pre_zlayer', 'bwd_type'])
                                .size().sort_values(ascending=False).head(10))
            for (fpre, fpost, bpre, bpost), n in flip_combos.items():
                print(f'  fwd {fpre}->{fpost}  ↔  bwd {bpre}->{bpost}: {n}')

    # Interactive controller
    _pair_cmap = widgets.Dropdown(options=CMAP_CHOICES, value='custom',
                                  description='Colours:')
    _pair_by_type = widgets.Checkbox(value=False, description='Colour scatter by type')
    _pair_diag = widgets.Checkbox(value=True, description='Show y=x diagonal')
    _pair_scat = widgets.Checkbox(value=True, description='Show scatter panel')
    _pair_mism = widgets.Checkbox(value=True, description='Show mismatch panel')
    _pair_zlayer = widgets.Checkbox(value=True, description='Show z-layer audit panel')

    def _go_pairs(**kw):
        _plot_pairs(pairs_df, kw['cmap'], kw['by_type'], kw['diag'],
                    kw['scat'], kw['mism'], show_zlayer=kw['zlayer'])

    _pair_controls = {'cmap': _pair_cmap, 'by_type': _pair_by_type,
                      'diag': _pair_diag, 'scat': _pair_scat,
                      'mism': _pair_mism, 'zlayer': _pair_zlayer}
    _pair_ui = widgets.VBox([
        widgets.HTML('<b>Interactive view</b> — toggle panels, colour scheme:'),
        widgets.HBox([_pair_cmap, _pair_by_type, _pair_diag]),
        widgets.HBox([_pair_scat, _pair_mism, _pair_zlayer]),
    ])
    display(_pair_ui, widgets.interactive_output(_go_pairs, _pair_controls))

In [ ]:
# Cell 8: Lattice Model Parameter Extraction
#
# For each move type, extract the parameters needed for an on-lattice KMC.

k0_THz = 10.0  # pyKMC attempt frequency in THz (= ps^-1)
k0_Hz = k0_THz * 1e12  # in Hz

print('=' * 70)
print('LATTICE MODEL PARAMETERS')
print('=' * 70)

params = {}
# Iterate over every known type (including z-layer-aware subsurface
# variants). ALL_TYPES is defined in 01_imports.py and stays in sync
# when new types are added.
for mt in ALL_TYPES:
    mt_data = sub[sub['move_type'] == mt]
    if len(mt_data) == 0:
        continue

    barriers = mt_data['energy_barrier']
    displacements = mt_data['move_dist']

    p = {
        'count': int(len(mt_data)),
        'Ea_mean_eV': float(barriers.mean()),
        'Ea_std_eV': float(barriers.std()),
        'Ea_min_eV': float(barriers.min()),
        'Ea_max_eV': float(barriers.max()),
        'displacement_mean_A': float(displacements.mean()),
        'displacement_std_A': float(displacements.std()),
        'prefactor_THz': k0_THz,
        'prefactor_Hz': k0_Hz,
        'nvac_range': [int(mt_data['nvac'].min()), int(mt_data['nvac'].max())],
        'temp_range': [int(mt_data['temp'].min()), int(mt_data['temp'].max())],
    }

    # Environment dependence: barrier vs nvac
    env_dep = mt_data.groupby('nvac')['energy_barrier'].agg(['mean', 'std', 'count'])
    p['barrier_by_nvac'] = env_dep.to_dict('index')

    params[mt] = p

    print(f'\n{mt}:')
    print(f'  Count: {p["count"]}')
    print(f'  Ea: {p["Ea_mean_eV"]:.4f} ± {p["Ea_std_eV"]:.4f} eV '
          f'(range: {p["Ea_min_eV"]:.4f} – {p["Ea_max_eV"]:.4f})')
    print(f'  Displacement: {p["displacement_mean_A"]:.3f} ± {p["displacement_std_A"]:.3f} A')
    print(f'  Prefactor: {k0_THz} THz')
    print(f'  Barrier by nvac:')
    for nv, stats in sorted(p['barrier_by_nvac'].items()):
        print(f'    {nv}vac: {stats["mean"]:.4f} ± {stats["std"]:.4f} eV (n={int(stats["count"])})')

# Lattice displacement vectors for FCC (100)
print(f'\n{"=" * 70}')
print('LATTICE DISPLACEMENT VECTORS (FCC 100 surface)')
print(f'{"=" * 70}')
print(f'\n1NN hop vectors (4 in-plane, a/sqrt(2) = {NN1_DIST:.3f} A):')
for dx, dy in [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
    v = np.array([dx, dy, 0]) * A_NI / 2
    print(f'  [{dx:+d},{dy:+d}, 0] x a/2 = [{v[0]:+.3f}, {v[1]:+.3f}, {v[2]:+.3f}] A')

print(f'\n2NN hop vectors (4 in-plane, a = {NN2_DIST:.3f} A):')
for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1)]:
    v = np.array([dx, dy, 0]) * A_NI
    print(f'  [{dx:+d},{dy:+d}, 0] x a = [{v[0]:+.3f}, {v[1]:+.3f}, {v[2]:+.3f}] A')

In [ ]:
# Cell 9: Validation — Compare analytical D to off-lattice MSD
#
# For 1vac simulations: D_lattice = f * Gamma * d^2 / 4
# where f = 0.4669 is the correlation factor for tracer diffusion via
# single vacancy on a 2D square lattice.
#
# NOTE: post-z-layer pass, `move_type == TYPE_1NN` now means a true
# surface 1NN hop (mover on top layer, both endpoints). Any bulk
# `subsurface_1NN_hop` events are excluded from this surface-diffusion
# validation, which is the physically correct behaviour.

from ase.io import read as ase_read

# Correlation factor for 2D square lattice tracer diffusion
f_corr = 0.4669

# Single-parameter model: use mean 1NN barrier across all 1vac simulations
ea_mean_all_1vac = sub[(sub['nvac'] == 1) & (sub['move_type'] == TYPE_1NN)]['energy_barrier'].mean()
print(f'Mean 1NN barrier across all 1vac sims: {ea_mean_all_1vac:.4f} eV')
print(f'Correlation factor: f = {f_corr}')
print(f'\nDiffusion coefficient validation (1vac only):')
print('-' * 70)
print(f'{"T(K)":>6s} {"Ea_1NN":>8s} {"D_MSD":>12s} {"D_uncorr":>12s} '
      f'{"D_corr":>12s} {"ratio_unc":>10s} {"ratio_corr":>10s}')

validation_rows = []
for sim in sims:
    if sim['nvac'] != 1:
        continue
    temp = sim['temp']
    sim_path = sim['path']

    # Load summary.json for MSD
    summary_path = sim_path / 'analysis' / 'summary.json'
    if not summary_path.exists():
        continue
    with open(summary_path) as f:
        sj = json.load(f)

    msd_final = sj.get('msd_final', [0])[0]
    total_time = sj.get('total_sim_time', 0)
    if total_time <= 0 or msd_final <= 0:
        continue

    # D from MSD: D = MSD / (2*d*t) where d=2 for 2D surface diffusion
    D_msd = msd_final / (4 * total_time)

    # D from per-simulation 1NN barrier
    sim_events = sub[(sub['nvac'] == 1) & (sub['temp'] == temp) & (sub['move_type'] == TYPE_1NN)]
    if len(sim_events) == 0:
        continue
    ea_1nn = sim_events['energy_barrier'].mean()

    rate = k0_Hz * np.exp(-ea_1nn / (kB * temp))
    D_uncorr = rate * NN1_DIST**2 / 4
    D_corr = f_corr * D_uncorr

    # Single-parameter model using mean barrier
    rate_mean = k0_Hz * np.exp(-ea_mean_all_1vac / (kB * temp))
    D_mean_uncorr = rate_mean * NN1_DIST**2 / 4
    D_mean_corr = f_corr * D_mean_uncorr

    ratio_uncorr = D_uncorr / D_msd if D_msd > 0 else float('inf')
    ratio_corr = D_corr / D_msd if D_msd > 0 else float('inf')

    validation_rows.append({
        'temp': temp,
        'Ea_1NN': ea_1nn,
        'D_msd': D_msd,
        'D_uncorr': D_uncorr,
        'D_corr': D_corr,
        'D_mean_uncorr': D_mean_uncorr,
        'D_mean_corr': D_mean_corr,
        'ratio_uncorr': ratio_uncorr,
        'ratio_corr': ratio_corr,
    })

    print(f'{temp:6d} {ea_1nn:8.4f} {D_msd:12.3e} {D_uncorr:12.3e} '
          f'{D_corr:12.3e} {ratio_uncorr:10.2f} {ratio_corr:10.2f}')

if validation_rows:
    vdf = pd.DataFrame(validation_rows).sort_values('temp')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Temperature plot
    ax = axes[0]
    ax.semilogy(vdf['temp'], vdf['D_msd'], 'ko-', label='D from MSD (off-lattice)', lw=2, markersize=8)
    ax.semilogy(vdf['temp'], vdf['D_uncorr'], 's--', color='#E41A1C',
                label='D lattice (uncorrected)', lw=1.5, markersize=6)
    ax.semilogy(vdf['temp'], vdf['D_corr'], '^-', color='#4DAF4A',
                label=f'D lattice (f={f_corr})', lw=2, markersize=8)
    ax.semilogy(vdf['temp'], vdf['D_mean_corr'], 'v:', color='#377EB8',
                label=f'D single-Ea model (Ea={ea_mean_all_1vac:.3f}, f={f_corr})', lw=1.5, markersize=6)
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('D (A^2/s)')
    ax.set_title('Surface 1NN hop diffusion validation (1vac, z-layer filtered)')
    ax.legend(fontsize=8)

    # Arrhenius plot
    ax = axes[1]
    inv_t = 1000.0 / vdf['temp']
    ax.plot(inv_t, np.log(vdf['D_msd']), 'ko-', label='D_MSD', lw=2, markersize=8)
    ax.plot(inv_t, np.log(vdf['D_corr']), '^-', color='#4DAF4A',
            label=f'D_lattice (f={f_corr})', lw=2, markersize=8)
    ax.plot(inv_t, np.log(vdf['D_mean_corr']), 'v:', color='#377EB8',
            label=f'Single-Ea model', lw=1.5, markersize=6)
    ax.set_xlabel('1000/T (1/K)')
    ax.set_ylabel('ln(D)')
    ax.set_title('Arrhenius plot')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'diffusion_validation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nSummary: corrected ratios range from '
          f'{vdf["ratio_corr"].min():.2f} to {vdf["ratio_corr"].max():.2f}')

In [ ]:
# Cell 10: Kinetic Coverage — per-simulation lattice step fraction
#
# For each simulation, compute what fraction of KMC steps are captured
# by on-lattice moves (1NN + 2NN hops).
#
# This cell uses the classifier path only: it reclassifies catalogue rows
# with classify_event_from_row() and then checks which selected ref_event
# values are typed as lattice moves. It does not read any event-level
# lattice projection export such as the deprecated mapped_1nn_events.csv.

coverage_rows = []

for sim in sims:
    sim_path = sim['path']
    pykmc_path = sim_path / 'pykmc.out'
    ref_path = sim_path / 'reference_table.pickle'

    if not pykmc_path.exists() or not ref_path.exists():
        continue

    df_steps = parse_pykmc_out(pykmc_path)
    if len(df_steps) == 0:
        continue

    catalog = pd.read_pickle(ref_path)

    # Reclassify each catalogue event with the shared displacement-based
    # classifier, then apply the z-layer pass so we mirror the semantics
    # of classify_lattice_events cell 07. We keep two accounting views:
    #   * "_any"     : geometry-only (pre z-layer) — old semantics, kept
    #                  for backward compatibility with downstream consumers
    #   * "_surface" : post z-layer — only true surface 1NN/2NN hops
    on_lattice_any = set()        # pre z-layer LATTICE_TYPES
    on_lattice_surface = set()    # post z-layer LATTICE_TYPES
    nn_hop_any = set()            # pre z-layer 1NN family
    nn_hop_surface = set()        # post z-layer 1NN family
    for i, row in catalog.iterrows():
        if row['energy_barrier'] >= BARRIER_CUTOFF:
            continue
        cls = classify_event_from_row(row)
        pre_t = cls['event_type']
        # z-layer relabel
        sl_i, sl_f = get_event_surface_layers(
            row['initial_positions'], row['final_positions'],
            int(row['move_atom_idx']))
        post_t = reclassify_by_zlayer(pre_t, sl_i, sl_f)
        if pre_t in LATTICE_TYPES:
            on_lattice_any.add(i)
        if post_t in LATTICE_TYPES:
            on_lattice_surface.add(i)
        if pre_t in {TYPE_1NN, TYPE_1NN_COOP}:
            nn_hop_any.add(i)
        if post_t in {TYPE_1NN, TYPE_1NN_COOP}:
            nn_hop_surface.add(i)

    total_steps = len(df_steps)
    total_time = df_steps['time'].iloc[-1] if len(df_steps) > 0 else 0

    def _pct(mask_set, col):
        if col == 'steps':
            n = df_steps['ref_event'].isin(mask_set).sum()
            return 100 * n / total_steps if total_steps > 0 else 0
        t = df_steps.loc[df_steps['ref_event'].isin(mask_set), 'dt'].sum()
        return 100 * t / total_time if total_time > 0 else 0

    coverage_rows.append({
        'nvac': sim['nvac'],
        'temp': sim['temp'],
        'total_steps': total_steps,
        'total_time': total_time,
        # Backward-compatible alias: _lattice = _lattice_any (pre z-layer)
        'pct_steps_lattice': _pct(on_lattice_any, 'steps'),
        'pct_steps_lattice_any': _pct(on_lattice_any, 'steps'),
        'pct_steps_lattice_surface': _pct(on_lattice_surface, 'steps'),
        'pct_steps_1nn': _pct(nn_hop_any, 'steps'),
        'pct_steps_1nn_any': _pct(nn_hop_any, 'steps'),
        'pct_steps_1nn_surface': _pct(nn_hop_surface, 'steps'),
        'pct_time_lattice': _pct(on_lattice_any, 'time'),
        'pct_time_lattice_any': _pct(on_lattice_any, 'time'),
        'pct_time_lattice_surface': _pct(on_lattice_surface, 'time'),
        'pct_time_1nn': _pct(nn_hop_any, 'time'),
        'pct_time_1nn_any': _pct(nn_hop_any, 'time'),
        'pct_time_1nn_surface': _pct(nn_hop_surface, 'time'),
    })

cov_df = pd.DataFrame(coverage_rows)
print('Kinetic coverage summary:')
print(cov_df.to_string(index=False, float_format='%.1f'))

if len(cov_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for col_idx, (col_any, col_surface, ylabel, title) in enumerate([
        ('pct_steps_lattice_any', 'pct_steps_lattice_surface',
         'Steps captured (%)', 'Kinetic coverage by steps (1NN + 2NN)'),
        ('pct_time_lattice_any', 'pct_time_lattice_surface',
         'Sim time captured (%)', 'Kinetic coverage by sim time (1NN + 2NN)'),
    ]):
        for row_idx, (col, suffix) in enumerate([(col_any, 'any (pre z-layer)'),
                                                 (col_surface, 'surface (post z-layer)')]):
            ax = axes[row_idx][col_idx]
            for nv in sorted(cov_df['nvac'].unique()):
                nv_data = cov_df[cov_df['nvac'] == nv].sort_values('temp')
                ax.plot(nv_data['temp'], nv_data[col],
                        'o-', label=f'{nv}vac', markersize=6)
            ax.set_xlabel('Temperature (K)')
            ax.set_ylabel(ylabel)
            ax.set_title(f'{title} — {suffix}')
            ax.legend(fontsize=7, ncol=2)
            ax.set_ylim(0, 105)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'kinetic_coverage.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 11: Export

# Derive composition from sim_path ({composition}/{N}vac/T{temp}/{variant},
# parsed by parse_run_path) so the event_viewer audit UI can filter across
# compositions even though one notebook run targets a single COMPOSITION.
# sim_relpath is sim_path relative to the data root, so the CSV still finds
# its runs on another machine (Analysis._paths.resolve_sim_path joins it to
# that machine's data root).
if 'sim_path' in sub.columns:
    sub = sub.copy()
    if 'composition' not in sub.columns:
        def _composition(p):
            parsed = parse_run_path(Path(p)) if p else None
            return parsed[0] if parsed else COMPOSITION
        sub['composition'] = sub['sim_path'].map(_composition)

    def _relpath(p):
        try:
            return Path(p).relative_to(DATA_ROOT.parent).as_posix()
        except (TypeError, ValueError):
            return ''
    sub['sim_relpath'] = sub['sim_path'].map(_relpath)

# 1. Classified events CSV (original + ML descriptors)
export_cols = [
    'composition', 'nvac', 'temp', 'sim_label', 'sim_path', 'sim_relpath',
    'idx_ref', 'energy_barrier', 'k',
    'move_type', 'move_type_base', 'move_type_pre_zlayer',
    'surface_layer_initial', 'surface_layer_final', 'zlayer_reclassified',
    'move_dist', 'move_dz', 'move_dz_signed', 'move_dxy', 'n_moved',
    'layer_change', 'idx_backward', 'dra',
    # --- ML Tier 1: mover-only ---
    'disp_top1', 'disp_top2', 'disp_top3', 'disp_top4', 'disp_top5',
    'mover_depth_initial', 'mover_depth_final', 'mover_depth_delta',
    'Ea_over_kT', 'mover_species_ml',
    'nn1_count_Ni', 'nn1_count_Fe', 'nn1_count_Cr',
    'nn2_count_Ni', 'nn2_count_Fe', 'nn2_count_Cr',
    # --- ML Tier 2: neighborhood geometry ---
    'coord_mover_initial', 'coord_mover_final',
    'n_vac_nn1_initial', 'n_vac_nn1_final',
    'mean_nn1_bond_length_initial', 'dist_to_nearest_undercoord',
    'local_rmsd_5A',
    # --- ML Tier 3: saddle-path ---
    'path_straightness', 'coord_mover_saddle',
    'bonds_broken_init_to_saddle', 'bonds_formed_init_to_saddle',
    # --- ML Tier 4: metadata ---
    'reverse_barrier', 'Ea_gap', 'saddle_asymmetry_oob', 'kmc_fire_count',
    # --- ML Tier 5: lattice-environment fingerprints ---
    'env_fingerprint_occ', 'env_fingerprint_species',
    'hop_vector_canonical', 'env_shell_complete',
    # --- 3D pipeline labels ---
    'motif_family_3d', 'site_class_3d', 'direction_family_3d',
]

# Compute 3D labels post-hoc from already-classified rows. The authoritative
# direction_family_3d lives in rate_lookup_table_3d_master.csv (built by
# build_rate_table.py with site-graph access); here we use a coarse fallback
# based on layer_change + displacement magnitudes.
from Analysis.lattice_map_events import classify_3d_motif
if 'motif_family_3d' not in sub.columns:
    sub['motif_family_3d'] = sub.apply(
        lambda r: classify_3d_motif({
            'move_type': r['move_type'],
            'n_moved': int(r.get('n_moved', 0) or 0),
            'surface_layer_initial': int(r.get('surface_layer_initial', -1) or -1),
        }),
        axis=1,
    )
if 'site_class_3d' not in sub.columns:
    def _site_class(li):
        try:
            li_int = int(li)
        except (TypeError, ValueError):
            return 'bulk_like'
        if li_int == 0:
            return 'surface'
        if li_int == 1:
            return 'subsurface'
        return 'bulk_like'
    sub['site_class_3d'] = sub['surface_layer_initial'].map(_site_class)
if 'direction_family_3d' not in sub.columns:
    # Canonical Cartesian classifier (same rule used by the runtime rate-table
    # builder and by the pylatkmc edge labeller). `classify_direction_xyz` only
    # uses np.hypot(dx, dy) and |dz|, so passing the in-plane magnitude as dx
    # with dy=0.0 is equivalent to passing the full Cartesian vector.
    from Analysis.lattice_3d import classify_direction_xyz
    def _vec_dir(r):
        dxy = float(r.get('move_dxy', 0.0) or 0.0)
        dz = float(r.get('move_dz_signed', r.get('move_dz', 0.0)) or 0.0)
        return classify_direction_xyz(dxy, 0.0, dz, NN1_DIST)
    sub['direction_family_3d'] = sub.apply(_vec_dir, axis=1)

# Subset to columns actually present (backward-compatible for older runs)
export_cols_present = [c for c in export_cols if c in sub.columns]
export_df = sub[export_cols_present].copy()
export_df.to_csv(OUT_DIR / 'classified_events.csv', index=False)
print(f'Saved classified_events.csv ({len(export_df)} events, {len(export_cols_present)} cols)')

# 1b. Enhanced classified events CSV (includes saddle features + enhanced classification)
enhanced_cols = export_cols + [
    'saddle_disp_mag', 'saddle_dz', 'saddle_height_ratio',
    'saddle_n_displaced', 'saddle_asymmetry', 'saddle_max_pair_dz',
    'enhanced_type', 'reclassified', 'original_event_type',
    'likely_reverse_relaxation', 'adaptive_type',
]
enhanced_present = [c for c in enhanced_cols if c in sub.columns]
enhanced_export = sub[enhanced_present].copy()
enhanced_export.to_csv(OUT_DIR / 'classified_events_enhanced.csv', index=False)
print(f'Saved classified_events_enhanced.csv ({len(enhanced_export)} events, {len(enhanced_present)} cols)')

# 2. Parameter summary JSON
param_summary = {
    'lattice': {
        'type': 'fcc_100_surface',
        'a_lat_A': A_NI,
        'nn1_dist_A': float(NN1_DIST),
        'nn2_dist_A': float(NN2_DIST),
    },
    'barrier_cutoff_eV': BARRIER_CUTOFF,
    'n_simulations': len(sims),
    'n_events_total': len(master),
    'n_events_classified': len(sub),
    'events_per_type': params,
    'enhanced_reclassified': int(sub['reclassified'].sum()) if 'reclassified' in sub.columns else 0,
    'zlayer_reclassified': int(sub['zlayer_reclassified'].sum()) if 'zlayer_reclassified' in sub.columns else 0,
}

# Add kinetic coverage
if len(cov_df) > 0:
    param_summary['kinetic_coverage'] = {
        'mean_pct_steps_lattice': float(cov_df['pct_steps_lattice'].mean()),
        'mean_pct_time_lattice': float(cov_df['pct_time_lattice'].mean()),
    }

with open(OUT_DIR / 'lattice_parameters.json', 'w') as f:
    json.dump(param_summary, f, indent=2, default=str)
print(f'Saved lattice_parameters.json')

# 3. Coverage CSV
if len(cov_df) > 0:
    cov_df.to_csv(OUT_DIR / 'kinetic_coverage.csv', index=False)
    print(f'Saved kinetic_coverage.csv')

print(f'\nAll outputs saved to {OUT_DIR}/')
print('Files:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name} ({p.stat().st_size / 1024:.1f} KB)')